# FlowEdit SD3 Midpoint Solver Evaluation

This notebook runs a controlled SD3 FlowEdit comparison using the best midpoint setting selected from the earlier sweep.

Experiment cases:

1. Lighthouse to Big Ben
2. Cat to small dog
3. Gas station sign to CVPR

Controlled variables:

- Same input images
- Same source and target prompts
- Same seed
- Same SD3 model
- Same `T_steps=16`, `n_min=0`, `n_max=16`, `src_guidance_scale=3.5`, `tar_guidance_scale=10.5`
- Only `solver_type` changes: `euler` vs `midpoint`

Evaluation:

- CLIP image-text alignment for target prompt matching
- DINO source-edited image similarity for structure preservation
- Runtime comparison
- Output image comparison table
- pandas summary table
- line charts
- t-SNE-style CLIP image embedding trajectory plot


In [ ]:
# 1) Run configuration
# Paste your Hugging Face token manually here before running the notebook.
# Do not commit a real token back to GitHub.
HF_TOKEN = "hf_your_token_here"

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "feature/midpoint-solver"
WORKDIR = "/content/FlowEdit"

EXP_YAML = "SD3_midpoint_best_controlled.yaml"
DATASET_YAML = "edits_midpoint_eval.yaml"
METRICS_DIR = "outputs/metrics"
PER_SAMPLE_CSV = f"{METRICS_DIR}/best16_clip_dino_per_sample.csv"
SUMMARY_CSV = f"{METRICS_DIR}/best16_clip_dino_summary.csv"
COMPARISON_CSV = f"{METRICS_DIR}/best16_solver_comparison_table.csv"
TSNE_CSV = f"{METRICS_DIR}/best16_clip_tsne_points.csv"

print("Repository:", REPO_URL)
print("Branch:", BRANCH)
print("Experiment YAML:", EXP_YAML)
print("Dataset YAML:", DATASET_YAML)


In [ ]:
# 2) Check GPU
!nvidia-smi


In [ ]:
# 3) Optional: mount Google Drive if you want outputs saved there
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# 4) Clone repository, checkout branch, and switch to workspace
from pathlib import Path

%cd /content
if not Path(WORKDIR).exists():
    !git clone {REPO_URL} {WORKDIR}

%cd {WORKDIR}
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git status --short --branch


In [ ]:
# 5) Install dependencies
# The first line keeps numpy, pandas, and scikit-learn binary wheels aligned.
# If this cell is run after importing numpy/pandas in the same runtime and an ABI error appears,
# restart the Colab runtime once and run again from Cell 1.
!pip install -q --upgrade --force-reinstall "numpy==2.0.2" "pandas==2.2.2" "scikit-learn==1.6.1"
!pip install -q --upgrade "plotly==5.24.1" "diffusers>=0.31.0" "transformers>=4.44.0" "accelerate>=0.33.0" "safetensors" "sentencepiece" "protobuf" "einops" "pyyaml" "pillow" "huggingface_hub"

try:
    import numpy as np
    import pandas as pd
    import sklearn
    import plotly
    print("numpy", np.__version__)
    print("pandas", pd.__version__)
    print("scikit-learn", sklearn.__version__)
    print("plotly", plotly.__version__)
except ValueError as exc:
    print("Package ABI error detected. In Colab, choose Runtime > Restart runtime, then run again from Cell 1.")
    raise


In [ ]:
# 6) Validate Hugging Face token
if not HF_TOKEN or HF_TOKEN.strip() in {"", "hf_your_token_here"}:
    raise ValueError("Paste your Hugging Face token into HF_TOKEN in Cell 1 before continuing.")

print("HF token is set in memory only.")


In [ ]:
# 7) Login to Hugging Face
from huggingface_hub import login

login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to Hugging Face for model download.")


In [ ]:
# 8) Check experiment files and dataset cases
from pathlib import Path
import yaml

required_files = [
    EXP_YAML,
    DATASET_YAML,
    "run_script.py",
    "FlowEdit_utils.py",
    "evaluate_clip_dino.py",
    "Data/Images/lighthouse.png",
    "Data/Images/cat.png",
    "Data/Images/gas_station.png",
]
missing = [p for p in required_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

print(f"Loaded {len(dataset)} edit cases:")
for i, item in enumerate(dataset, start=1):
    print(f"{i}. {item['input_img']} -> {item['target_codes'][0]}")
    print("   Target:", item["target_prompts"][0])


In [ ]:
# 9) Run controlled best-16 Euler vs Midpoint experiment, then evaluate CLIP + DINO
import shutil
import subprocess
from pathlib import Path

cleanup_paths = [
    "outputs/Best16_Control_Euler",
    "outputs/Best16_Control_Midpoint",
    "outputs/run_summary.csv",
    PER_SAMPLE_CSV,
    SUMMARY_CSV,
    COMPARISON_CSV,
    TSNE_CSV,
]
for path in cleanup_paths:
    p = Path(path)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)

subprocess.run(["python", "run_script.py", "--device_number", "0", "--exp_yaml", EXP_YAML], check=True)
subprocess.run([
    "python", "evaluate_clip_dino.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--dataset_yaml", DATASET_YAML,
    "--out_samples", PER_SAMPLE_CSV,
    "--out_summary", SUMMARY_CSV,
], check=True)

if not Path(PER_SAMPLE_CSV).exists() or not Path(SUMMARY_CSV).exists():
    raise FileNotFoundError("Evaluation finished without writing metric CSV files. Check the logs above.")

print("\nSummary metrics:")
print(Path(SUMMARY_CSV).read_text())


In [ ]:
# 10) Build an output image comparison table with pandas
import base64
import html
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import HTML, display

if not Path(PER_SAMPLE_CSV).exists():
    raise FileNotFoundError(f"Missing {PER_SAMPLE_CSV}. Run Cell 9 successfully before this visualization cell.")

metrics = pd.read_csv(PER_SAMPLE_CSV)
metrics["clip_alignment"] = metrics["clip_alignment"].astype(float)
metrics["dino_similarity"] = metrics["dino_similarity"].astype(float)
metrics["elapsed_seconds"] = metrics["elapsed_seconds"].astype(float)

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)


def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "")
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"


def img_tag(path, width=260):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border-radius:8px;border:1px solid #ddd;'>"

rows_html = []
for item in dataset:
    source = item["input_img"]
    code = item["target_codes"][0]
    target = item["target_prompts"][0]
    subset = metrics[metrics["source_image"] == source]
    euler = subset[subset["solver_type"].str.lower() == "euler"].iloc[0]
    midpoint = subset[subset["solver_type"].str.lower() == "midpoint"].iloc[0]

    rows_html.append(f"""
    <tr>
      <td><b>{html.escape(code)}</b><br><span class='small'>{html.escape(target)}</span></td>
      <td>{img_tag(source)}</td>
      <td>{img_tag(euler['output_image'])}<br><b>Euler</b><br>CLIP {euler['clip_alignment']:.4f} | DINO {euler['dino_similarity']:.4f}<br>{euler['elapsed_seconds']:.1f}s</td>
      <td>{img_tag(midpoint['output_image'])}<br><b>Midpoint</b><br>CLIP {midpoint['clip_alignment']:.4f} | DINO {midpoint['dino_similarity']:.4f}<br>{midpoint['elapsed_seconds']:.1f}s</td>
    </tr>
    """)

html_table = f"""
<style>
.compare-table {{ border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; }}
.compare-table th, .compare-table td {{ border: 1px solid #ddd; padding: 10px; vertical-align: top; text-align: center; }}
.compare-table th {{ background: #f4f4f4; }}
.compare-table td:first-child {{ text-align: left; width: 24%; }}
.small {{ color: #555; font-size: 12px; line-height: 1.35; }}
</style>
<h3>Best-16 Controlled Output Comparison</h3>
<table class='compare-table'>
  <tr><th>Case and Target Prompt</th><th>Input</th><th>Euler</th><th>Midpoint</th></tr>
  {''.join(rows_html)}
</table>
"""

display(HTML(html_table))


In [ ]:
# 11) Build a quantitative solver comparison table
from pathlib import Path

import pandas as pd
from IPython.display import display

if not Path(PER_SAMPLE_CSV).exists():
    raise FileNotFoundError(f"Missing {PER_SAMPLE_CSV}. Run Cell 9 successfully before this visualization cell.")

metrics = pd.read_csv(PER_SAMPLE_CSV)
metrics["case"] = metrics["source_image"].map(lambda p: Path(p).stem)
for col in ["clip_alignment", "dino_similarity", "elapsed_seconds"]:
    metrics[col] = metrics[col].astype(float)
metrics["edit_preservation_score"] = metrics["clip_alignment"] * metrics["dino_similarity"]

wide = metrics.pivot(index="case", columns="solver_type", values=[
    "clip_alignment",
    "dino_similarity",
    "elapsed_seconds",
    "edit_preservation_score",
])
wide.columns = [f"{metric}_{solver}" for metric, solver in wide.columns]
wide = wide.reset_index()

wide["clip_midpoint_minus_euler"] = wide["clip_alignment_midpoint"] - wide["clip_alignment_euler"]
wide["dino_midpoint_minus_euler"] = wide["dino_similarity_midpoint"] - wide["dino_similarity_euler"]
wide["score_midpoint_minus_euler"] = wide["edit_preservation_score_midpoint"] - wide["edit_preservation_score_euler"]
wide["runtime_midpoint_over_euler"] = wide["elapsed_seconds_midpoint"] / wide["elapsed_seconds_euler"]
wide["winner_by_score"] = wide["score_midpoint_minus_euler"].map(lambda x: "midpoint" if x > 0 else "euler")

wide.to_csv(COMPARISON_CSV, index=False)
print("Wrote:", COMPARISON_CSV)
display(wide.style.format({
    "clip_alignment_euler": "{:.4f}",
    "clip_alignment_midpoint": "{:.4f}",
    "dino_similarity_euler": "{:.4f}",
    "dino_similarity_midpoint": "{:.4f}",
    "elapsed_seconds_euler": "{:.1f}",
    "elapsed_seconds_midpoint": "{:.1f}",
    "edit_preservation_score_euler": "{:.4f}",
    "edit_preservation_score_midpoint": "{:.4f}",
    "clip_midpoint_minus_euler": "{:+.4f}",
    "dino_midpoint_minus_euler": "{:+.4f}",
    "score_midpoint_minus_euler": "{:+.4f}",
    "runtime_midpoint_over_euler": "{:.2f}x",
}))


In [ ]:
# 12) Draw line charts for CLIP, DINO, edit-preservation score, and runtime
from pathlib import Path
import pandas as pd
import plotly.express as px

if not Path(PER_SAMPLE_CSV).exists():
    raise FileNotFoundError(f"Missing {PER_SAMPLE_CSV}. Run Cell 9 successfully before this visualization cell.")

metrics = pd.read_csv(PER_SAMPLE_CSV)
metrics["case"] = metrics["source_image"].map(lambda p: Path(p).stem)
for col in ["clip_alignment", "dino_similarity", "elapsed_seconds"]:
    metrics[col] = metrics[col].astype(float)
metrics["edit_preservation_score"] = metrics["clip_alignment"] * metrics["dino_similarity"]
metrics["solver_type"] = pd.Categorical(metrics["solver_type"], categories=["euler", "midpoint"], ordered=True)
metrics = metrics.sort_values(["case", "solver_type"])

chart_specs = [
    ("clip_alignment", "CLIP target-prompt alignment (higher is better)"),
    ("dino_similarity", "DINO source preservation (higher is more structure-preserving)"),
    ("edit_preservation_score", "CLIP x DINO balanced score (higher is better)"),
    ("elapsed_seconds", "Runtime seconds (lower is faster)"),
]

for metric, title in chart_specs:
    fig = px.line(
        metrics,
        x="case",
        y=metric,
        color="solver_type",
        markers=True,
        title=title,
        category_orders={"solver_type": ["euler", "midpoint"]},
    )
    fig.update_layout(width=850, height=420, template="plotly_white")
    fig.show()


In [ ]:
# 13) t-SNE-style CLIP image embedding trajectory plot
# This shows where the input, Euler output, and Midpoint output land in CLIP image-embedding space.
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from sklearn.manifold import TSNE
from transformers import CLIPModel, CLIPProcessor

if not Path(PER_SAMPLE_CSV).exists():
    raise FileNotFoundError(f"Missing {PER_SAMPLE_CSV}. Run Cell 9 successfully before this visualization cell.")

metrics = pd.read_csv(PER_SAMPLE_CSV)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

records = []
for item in dataset:
    source = item["input_img"]
    case = item["target_codes"][0]
    subset = metrics[metrics["source_image"] == source]
    euler = subset[subset["solver_type"].str.lower() == "euler"].iloc[0]
    midpoint = subset[subset["solver_type"].str.lower() == "midpoint"].iloc[0]
    records.extend([
        {"case": case, "stage": "input", "path": source, "solver_type": "input"},
        {"case": case, "stage": "euler", "path": euler["output_image"], "solver_type": "euler"},
        {"case": case, "stage": "midpoint", "path": midpoint["output_image"], "solver_type": "midpoint"},
    ])

images = [Image.open(r["path"]).convert("RGB") for r in records]
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device).eval()

dummy_text = ["image"] * len(images)
inputs = processor(text=dummy_text, images=images, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    outputs = model(**inputs)
    feats = outputs.image_embeds
feats = F.normalize(feats, dim=-1).cpu().numpy()

perplexity = min(3, len(records) - 1)
coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=42,
).fit_transform(feats)

tsne_df = pd.DataFrame(records)
tsne_df["x"] = coords[:, 0]
tsne_df["y"] = coords[:, 1]
tsne_df.to_csv(TSNE_CSV, index=False)
print("Wrote:", TSNE_CSV)

stage_order = {"input": 0, "euler": 1, "midpoint": 2}
fig = go.Figure()
for case, group in tsne_df.groupby("case"):
    group = group.sort_values("stage", key=lambda s: s.map(stage_order))
    fig.add_trace(go.Scatter(
        x=group["x"],
        y=group["y"],
        mode="lines+markers+text",
        text=group["stage"],
        textposition="top center",
        name=case,
        hovertext=group["path"],
        hoverinfo="text+name",
    ))

fig.update_layout(
    title="t-SNE-style CLIP embedding trajectories: input -> Euler -> Midpoint",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
    width=900,
    height=650,
    template="plotly_white",
)
fig.show()


## How to Read the Results

- Higher CLIP alignment usually means the edited image matches the target prompt better.
- Higher DINO similarity usually means the edited image preserves more source-image structure.
- Midpoint is more expensive per solver step, so runtime may still be slower even when the image is better.
- For editing, the best output is usually not the highest CLIP alone. A useful result should improve the target concept while keeping enough DINO similarity to preserve the original layout.
- The t-SNE plot is qualitative. It helps visualize whether Midpoint lands in a different image-embedding neighborhood than Euler, but the CLIP/DINO table should be the main quantitative evidence.

## Troubleshooting

- If pandas raises `numpy.dtype size changed`, restart the Colab runtime once and run again from Cell 1. This happens when binary packages are changed after imports.
- If SD3 download fails, check that `HF_TOKEN` has access to `stabilityai/stable-diffusion-3-medium-diffusers`.
- If CUDA runs out of memory, switch Colab to an L4/A100 GPU or lower image resolution in the source images.
